# 02 — Retrieval Engines Sanity Check

Builds all three retrieval engines (BM25, Vector, Hybrid) and verifies they
work correctly by running a sanity-check query side-by-side.

**Prerequisites:** Run `01_data_collection.ipynb` first to generate `../data/chunks.json`.

**Purpose:** Validate that all three engines return results in the expected schema
before running the full experiment in `03_experiment.ipynb`.

In [1]:
# Cell 1 — Install dependencies and imports
!pip install rank-bm25 sentence-transformers faiss-cpu

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.chunker import load_chunks
from src.bm25_retriever import BM25Retriever
from src.vector_retriever import VectorRetriever
from src.hybrid_retriever import HybridRetriever

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 8.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/11.2 MB 9.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.4/11.2 MB 5.8 MB/s eta 0:00:02
   -------- ------------------------------- 2.4/11.2 MB 5.8 MB/s eta 0:00:02
   -------------- ------------------------- 3.9/11.2 MB 5.0 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.2 MB 6.2 MB/s eta 0:00:01
   ------------------------------- -------- 8.7/11.2 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 7.5 MB/s eta 0:00:00
   --------------------------------------

In [2]:
# Cell 2 — Load data
chunks = load_chunks("../data/chunks.json")
print(f"Loaded {len(chunks)} chunks")

Loaded 1918 chunks from ../data/chunks.json
Loaded 1918 chunks


In [3]:
# Cell 3 — Build all indexes
bm25 = BM25Retriever()
bm25.build_index(chunks)

vector = VectorRetriever()
vector.build_index(chunks)

hybrid = HybridRetriever(bm25, vector)

BM25 index built: 1918 chunks
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\userPC\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\userPC\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index built: 1918 chunks, dim=384


In [4]:
# Cell 4 — Run sanity-check query and display side-by-side
TEST_QUERY = "Who is the best player in the 2026 FIFA World Cup?"
TOP_K = 3

bm25_results   = bm25.search(TEST_QUERY, top_k=TOP_K)
vector_results = vector.search(TEST_QUERY, top_k=TOP_K)
hybrid_results = hybrid.search(TEST_QUERY, top_k=TOP_K)

print(f"Query: '{TEST_QUERY}'")
print()

# Print side-by-side table
col_w = 42  # characters per column
header = f"{'Rank':<5} | {'BM25':<{col_w}} | {'Vector':<{col_w}} | {'Hybrid':<{col_w}}"
sep    = "-" * len(header)
print(header)
print(sep)

for i in range(TOP_K):
    def fmt(results, idx):
        if idx >= len(results):
            return "(none)"
        r = results[idx]
        cell = f"[{r['source'][:18]}] {r['text'][:20]}..."
        return cell[:col_w]

    print(f"{i+1:<5} | {fmt(bm25_results, i):<{col_w}} | {fmt(vector_results, i):<{col_w}} | {fmt(hybrid_results, i):<{col_w}}")

print()
print("Full top-1 results:")
print(f"  BM25   score={bm25_results[0]['score']:.4f}  | {bm25_results[0]['source']}")
print(f"    {bm25_results[0]['text'][:120]} ...")
print(f"  Vector score={vector_results[0]['score']:.4f}  | {vector_results[0]['source']}")
print(f"    {vector_results[0]['text'][:120]} ...")
print(f"  Hybrid rrf={hybrid_results[0]['rrf_score']:.5f} | bm25_rank={hybrid_results[0]['bm25_rank']} | vec_rank={hybrid_results[0]['vector_rank']} | {hybrid_results[0]['source']}")
print(f"    {hybrid_results[0]['text'][:120]} ...")

Query: 'Who is the best player in the 2026 FIFA World Cup?'

Rank  | BM25                                       | Vector                                     | Hybrid                                    
--------------------------------------------------------------------------------------------------------------------------------------------
1     | [2026 FIFA World Cu] The 2026 FIFA World . | [Lionel Messi] Reception
Messi is w...     | [Kylian Mbappé] Former French intern...   
2     | [FIFA] Recognition and awar...             | [Vinícius Júnior] Copa São Paulo de Fu...  | [Lionel Messi] Reception
Messi is w...    
3     | [Kylian Mbappé] Mbappé, who scored a...    | [Lionel Messi] Widely regarded as o...     | [Vinícius Júnior] Copa São Paulo de Fu... 

Full top-1 results:
  BM25   score=20.6936  | 2026 FIFA World Cup
    The 2026 FIFA World Cup is the current and 23rd edition of the FIFA World Cup, the quadrennial international men's socce ...
  Vector score=0.6263  | Lionel Messi


In [5]:
# Cell 5 — Verify output schema
BM25_REQUIRED_KEYS   = {'rank', 'score', 'chunk_id', 'source', 'text'}
VECTOR_REQUIRED_KEYS = {'rank', 'score', 'chunk_id', 'source', 'text'}
HYBRID_REQUIRED_KEYS = {'rank', 'rrf_score', 'bm25_rank', 'vector_rank', 'chunk_id', 'source', 'text'}

for i, result in enumerate(bm25_results):
    missing = BM25_REQUIRED_KEYS - set(result.keys())
    assert not missing, f"BM25 result {i} missing keys: {missing}"

for i, result in enumerate(vector_results):
    missing = VECTOR_REQUIRED_KEYS - set(result.keys())
    assert not missing, f"Vector result {i} missing keys: {missing}"

for i, result in enumerate(hybrid_results):
    missing = HYBRID_REQUIRED_KEYS - set(result.keys())
    assert not missing, f"Hybrid result {i} missing keys: {missing}"

# Verify rank is 1-indexed
assert bm25_results[0]['rank'] == 1,   "BM25 rank should start at 1"
assert vector_results[0]['rank'] == 1, "Vector rank should start at 1"
assert hybrid_results[0]['rank'] == 1, "Hybrid rank should start at 1"

print("All schema checks passed")

All schema checks passed
